**Preguntas a un LLM sobre datos propios**

Este programa permite consultar datos textuales propios a un LLM basado en comparación de embeddings (*pregunta-respuesta o QA*). Para esto, los datos a consultar se convierten transforman en base de datos de vectores que almacenan eficientemente los embeddings de los textos. Estas bases de datos están diseñadas para realizar búsquedas por similitud en espacios altamente dimensionales, posibilitando la recuperación de los resultados más semánticamente relevantes.

Al almacenar embeddings de documentos (o párrafos) en una base de datos de vectores, un sistema de búsqueda puede rápidamente identificar los textos  que mejor calzan con una consulta (*query*).

Para este ejemplo, se utilizará  el framework [LangChain](https://python.langchain.com/) de OpenAI para desarrollar aplicaciones simples basadas en LLMs. LangChain permite conectar un LLM a otras fuentes de datos (i.e., bases de datos SQL, buscador de Google, etc).

Primero, instalamos los paquetes necesarios de LangChain, OpenAI, y otros

In [ ]:
!pip list

Package                               Version
------------------------------------- -------------------
absl-py                               1.4.0
accelerate                            1.6.0
aiohappyeyeballs                      2.6.1
aiohttp                               3.11.15
aiosignal                             1.3.2
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.6
ale-py                                0.11.0
altair                                5.5.0
annotated-types                       0.7.0
antlr4-python3-runtime                4.9.3
anyio                                 4.9.0
argon2-cffi                           23.1.0
argon2-cffi-bindings                  21.2.0
array_record                          0.7.2
arviz                                 0.21.0
astropy                               7.0.2
astropy-iers-data                     0.2025.5.12.0.38.29
astunparse                            1

In [1]:
!pip install duckdb
!pip install unstructured
!pip install chromadb
!pip install BeautifulSoup4
#!pip install openai
!pip install tiktoken
!pip install langchain_community
!pip install -U langchain-unstructured
!pip install langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.6/167.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 13.7 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=95d5dad36fcf31f759b6b32ac2520444693082d5d69f9c0ea52a4cee834e8803
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4

Importamos algunas librerías relevantes para obtener embeddings y consultar o recuperar (*Retrieve*) desde la bases de datos de embeddings:

In [2]:
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_unstructured import UnstructuredLoader
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
import openai
import os
import tiktoken

Asignamos la clave para poder acceder a la API de un LLM:

In [3]:
# Open AI API-key
from google.colab import files
from IPython.display import clear_output

files.upload() # subir archivo con apikey de openai propio
clear_output() # no muestra contenido del apikey

In [4]:
def get_api_key():
    with open('idsa_openai_key.txt', 'r') as fp: #acá reemplazar x el nombre de tu archivo
        key = fp.read()
    return key

# Enter your OpenAI API key here
openai_api_key = get_api_key()

In [5]:
!wget https://github.com/palasatenea66/DATASETS/raw/main/funes.txt

--2025-08-15 00:32:15--  https://github.com/palasatenea66/DATASETS/raw/main/funes.txt
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/palasatenea66/DATASETS/main/funes.txt [following]
--2025-08-15 00:32:15--  https://raw.githubusercontent.com/palasatenea66/DATASETS/main/funes.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16021 (16K) [text/plain]
Saving to: ‘funes.txt’

funes.txt           100%[===================>]  15.65K  --.-KB/s    in 0.001s  

2025-08-15 00:32:15 (21.3 MB/s) - ‘funes.txt’ saved [16021/16021]



Cargamos un texto de prueba ("funes.txt"):

In [6]:
!ls

funes.txt  idsa_openai_key.txt	sample_data


In [7]:
loader = UnstructuredLoader('funes.txt')
documentos = loader.load()

In [8]:
# Modify the metadata
for document in documentos:
    document.metadata['languages'] = 'spa'

Ahora, separamos el texto en trozos (*chunks*) de 1000 caracteres (sin sobreponer chunks: *chunk_overlap=0*):

In [9]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
MisTextos = text_splitter.split_documents(documentos)

Inicializamos la función que calculará los embeddings:

In [10]:
Mis_Embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

Creamos una base de datos de vectores a partir de nuestros datos y la utilizamos para indexar los embeddings:

In [11]:
db = Chroma.from_documents(MisTextos, Mis_Embeddings)

Nuestros datos ya están indexados (i.e., párrafo con sus correspondientes IDs y embeddings) para poder consultar sobre ellos. Para esto, utilizamos la función **RetrievalQA** de *LangChain* que inicializa nuestro framework LangChain con el LLM disponible por defecto ("*text-embedding-ada-002*") y nuestra base de datos de vectores (*db*). Note que sólo se permite una respuesta de salida (*k=1*):

In [12]:
from langchain_openai import ChatOpenAI
qa = RetrievalQA.from_chain_type(
              llm=ChatOpenAI(openai_api_key=openai_api_key),
              chain_type="stuff",
              retriever=db.as_retriever(search_kwargs={"k": 1}))

Ahora, podemos preguntar a nuestro modelo. Para esto, internamente se convierte la consulta a su embedding, y se busca similitudes entre este y los disponibles en la base de datos de vectores de nuestros propios datos:

In [13]:
query = "¿De qué se trata el documento?"
qa.invoke(query)

{'query': '¿De qué se trata el documento?',
 'result': 'El documento se trata de un relato literario titulado "Funes el memorioso" escrito por el autor Jorge Luis Borges. El relato narra la historia de un personaje llamado Ireneo Funes, quien sufría de una memoria prodigiosa después de un accidente que lo dejó paralítico. Funes podía recordar absolutamente cada detalle de su vida y entorno, lo que lo hizo percibir el mundo de una manera mucho más intensa y detallada que la mayoría de las personas. La historia explora las complejidades y dificultades de tener una memoria tan excepcional.'}

In [14]:
query = "¿Quién es Ireneo Funes?"
qa.invoke(query)

{'query': '¿Quién es Ireneo Funes?',
 'result': 'Ireneo Funes es un personaje ficticio creado por el escritor Jorge Luis Borges en su relato "Funes el memorioso". Funes es un joven uruguayo con una memoria prodigiosa que posee la capacidad de recordar absolutamente todo lo que ha visto, escuchado o experimentado en su vida. Esta habilidad sobrehumana lo lleva a una existencia solitaria y atormentada, ya que su memoria perfecta no le permite generalizar, abstraerse o pensar de manera abstracta. La historia de Funes el memorioso trata sobre las limitaciones y paradojas de la memoria y la percepción humana.'}

In [15]:
query = "¿Quién es Borges?"
qa.invoke(query)

{'query': '¿Quién es Borges?',
 'result': 'Jorge Luis Borges fue un escritor argentino, considerado uno de los escritores más importantes en lengua española del siglo XX. Sus escritos abarcan una amplia gama de géneros, incluyendo ensayos, cuentos y poesía. Sus obras son conocidas por su estilo innovador, complejidad intelectual y referencias literarias y filosóficas sofisticadas. Borges es especialmente reconocido por su habilidad para mezclar la realidad y la ficción, así como por explorar temas como la identidad, el tiempo, los laberintos y la metaficción.'}

In [16]:
query = "¿Qué cuentos escribió Borges?"
qa.invoke(query)

{'query': '¿Qué cuentos escribió Borges?',
 'result': 'Jorge Luis Borges escribió una extensa variedad de cuentos a lo largo de su carrera. Algunos de sus cuentos más famosos son "El jardín de senderos que se bifurcan", "Pierre Menard, autor del Quijote", "La biblioteca de Babel", "Funes el memorioso", entre otros. Borges es conocido por su trabajo innovador y filosófico en el campo del cuento corto y de la literatura en general.'}

In [17]:
query='¿Por qué razón Funes quedó paralítico?'
qa.invoke(query)

{'query': '¿Por qué razón Funes quedó paralítico?',
 'result': 'Funes quedó paralítico después de caerse de un caballo y golpear su cabeza contra el suelo. Este incidente lo llevó a desarrollar una memoria prodigiosa y una percepción descomunal de la realidad y sus detalles.'}

In [18]:
query='¿En qué año conoció Funes a Emma Zunz?'
qa.invoke(query)

{'query': '¿En qué año conoció Funes a Emma Zunz?',
 'result': 'No se encontró ninguna referencia a un encuentro entre Funes e Emma Zunz en el texto proporcionado. Por lo tanto, no es posible determinar en qué año conoció Funes a Emma Zunz.'}